In [2]:
%load_ext autoreload
%autoreload 2
import os
import pickle
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
from copy import deepcopy

# import torch
# from esm.models.esmc import ESMC
# from esm.tokenization import EsmSequenceTokenizer
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Predict CASP15 contacts

In [4]:
# Load target data pickle file
target_data_file = "../../data/Figure5_CASP_ProteinGym/target_data.pkl"
with open(target_data_file, "rb") as oFile:
    target_data_d = pickle.load(oFile)    

## 1.1 ESM C 300M

In [10]:
# Load model
head_out_dir = "../notebook_outs/09h_fit_esmc_contact_heads/esmc_300m/"
model = ESMC.from_pretrained("esmc_300m", use_flash_attn=False, init_contact_head=True, contact_head_weights_path=os.path.join(head_out_dir, "esmc_300m_contact_heads.pt")).to(device)
model.eval()
print("Successfully loaded model")
# Print number of parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {total_params:,}")

# Load tokenizer
tokenizer = EsmSequenceTokenizer()

True
Successfully loaded model
Number of parameters: 332,997,635


In [ ]:
all_results_d = {}
results_dir = "results/esmc_300m/"

# Initialize results dictionaries
res_d = {}
pred_contacts_d = {}

# Iterate over all targets
for target_id in tqdm(target_data_d):
    # Load MSA
    msa_file = target_data_d[target_id]["msa_file"]
    with open(msa_file, "r") as oFile:
        sequence = oFile.readlines()[1].strip()
    if len(sequence) > 2048:
        print(f"Skipping {target_id} because it has more than 2048 tokens...rerun with half precision: {target_id}")
        continue

    # Predict contacts
    tokens = tokenizer.encode(sequence, return_tensors="pt").to(device)
    with torch.no_grad():
        # Compute attention heads
        output = model(sequence_tokens=tokens, output_attentions=True, return_contacts=True)
        pred_contacts_a = output.contacts[0]
        
    # Load contacts
    contacts_a = target_data_d[target_id]["contacts_a"]
    # Mask for valid positions
    valid_mask = contacts_a >= 0
    contacts_a[~valid_mask] = 0
    msa_subset_idx_l = target_data_d[target_id]["msa_subset_idx_l"]
    # Run predictions
    subset_contacts_a = deepcopy(pred_contacts_a[msa_subset_idx_l, :][:, msa_subset_idx_l])
    subset_contacts_a[~valid_mask] = 0
    pred_contacts_d[target_id] = subset_contacts_a

pred_contacts_path = os.path.join(results_dir, f"all_pred_contacts_CASP15.pkl")
with open(pred_contacts_path, "wb") as oFile:
    pickle.dump(pred_contacts_d, oFile)
results_path = "results/esmc_300m/all_results_CASP15.tsv"
pd.DataFrame(res_d).T.to_csv(results_path, sep="\t")

  0%|          | 0/49 [00:00<?, ?it/s]

 39%|███▉      | 19/49 [00:22<00:23,  1.26it/s]

Skipping T1169 because it has more than 2048 tokens...rerun with half precision: T1169


100%|██████████| 49/49 [00:33<00:00,  1.45it/s]


## 1.2 ESM C 600M

In [24]:
# Load model
head_out_dir = "../notebook_outs/09h_fit_esmc_contact_heads/esmc_600m/"
model = ESMC.from_pretrained("esmc_600m", use_flash_attn=False, init_contact_head=True, contact_head_weights_path=os.path.join(head_out_dir, "esmc_600m_contact_heads.pt")).to(device)
model.eval()
print("Successfully loaded model")
# Print number of parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {total_params:,}")

# Load tokenizer
tokenizer = EsmSequenceTokenizer()

Successfully loaded model
Number of parameters: 575,037,641


In [ ]:
all_results_d = {}
results_dir = "results/esmc_600m/"

# Initialize results dictionaries
pred_contacts_d = {}

# Iterate over all targets
for target_id in tqdm(target_data_d):
    # Load MSA
    msa_file = target_data_d[target_id]["msa_file"]
    with open(msa_file, "r") as oFile:
        sequence = oFile.readlines()[1].strip()
    if len(sequence) > 2048:
        print(f"Skipping {target_id} because it has more than 2048 tokens...rerun with half precision: {target_id}")
        continue

    # Predict contacts
    tokens = tokenizer.encode(sequence, return_tensors="pt").to(device)
    with torch.no_grad():
        # Compute attention heads
        output = model(sequence_tokens=tokens, output_attentions=True, return_contacts=True)
        pred_contacts_a = output.contacts[0]
        
    # Load contacts
    contacts_a = target_data_d[target_id]["contacts_a"]
    # Mask for valid positions
    valid_mask = contacts_a >= 0
    contacts_a[~valid_mask] = 0
    msa_subset_idx_l = target_data_d[target_id]["msa_subset_idx_l"]
    # Run and evaluate predictions
    subset_contacts_a = deepcopy(pred_contacts_a[msa_subset_idx_l, :][:, msa_subset_idx_l])
    subset_contacts_a[~valid_mask] = 0
    pred_contacts_d[target_id] = subset_contacts_a

pred_contacts_path = os.path.join(results_dir, f"all_pred_contacts_CASP15.pkl")
with open(pred_contacts_path, "wb") as oFile:
    pickle.dump(pred_contacts_d, oFile)
results_path = "results/esmc_600m/all_results_CASP15.tsv"
pd.DataFrame(res_d).T.to_csv(results_path, sep="\t")

  0%|          | 0/49 [00:00<?, ?it/s]

 37%|███▋      | 18/49 [00:07<00:09,  3.26it/s]

Skipping T1169 because it has more than 2048 tokens...rerun with half precision: T1169


100%|██████████| 49/49 [00:12<00:00,  3.88it/s]
